## ÉTAPE 13 — Classification (Random Forest)
Objectif : prédire si un pays est "grand producteur"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv("JEU_DE_DONNEE.csv")

cultures = [
    'cereals_total', 'roots_and_tubers_total', 'vegetables_melons_total',
    'fruit_excl_melons_total', 'oilcrops_primary', 'pulses_total',
    'wheat', 'maize', 'rice_paddy', 'sugar_cane'
]


## 1. PRÉPARER LES FEATURES (X) ET LA CIBLE (y)


In [ ]:
df_pays = df[
    (df["element"] == "Production Quantity") &
    (~df["country_or_area"].str.contains(r"\+", na=False))
].dropna(subset=["value"])

tableau = (
    df_pays[df_pays["category"].isin(cultures)]
    .groupby(["country_or_area", "category"])["value"]
    .sum()
    .reset_index()
    .pivot_table(index="country_or_area", columns="category", values="value", aggfunc="sum")
    .fillna(0)
)

# Cible : 1 = grand producteur (production totale > médiane), 0 = petit
total_par_pays = tableau.sum(axis=1)
mediane = total_par_pays.median()
y = (total_par_pays > mediane).astype(int)
X = tableau.values

print(f"Pays analysés     : {len(tableau)}")
print(f"Grands producteurs: {y.sum()} | Petits : {(y==0).sum()}")


## 2. ENTRAÎNER LE MODÈLE


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("\n--- Rapport de classification ---")
print(classification_report(y_test, y_pred,
      target_names=["Petit producteur", "Grand producteur"]))


## 3. GRAPHIQUE 1 — MATRICE DE CONFUSION


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Petit", "Grand"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matrice de confusion", fontsize=12, fontweight="bold")


## 4. GRAPHIQUE 2 — IMPORTANCE DES FEATURES


In [ ]:
importances = pd.Series(rf.feature_importances_, index=cultures).sort_values()

importances.plot(kind="barh", ax=axes[1], color="#2563EB", alpha=0.85)
axes[1].set_title("Importance des cultures pour la prédiction",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Importance")
axes[1].grid(axis="x", alpha=0.4)

plt.suptitle("Random Forest — Classification des pays agricoles",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("random_forest.png", dpi=150, bbox_inches="tight")
print("\nGraphique Random Forest sauvegardé ✓")
plt.show()

print("\n=== RÉSUMÉ RANDOM FOREST ===")
print("RandomForestClassifier()  → créer le modèle")
print(".fit(X_train, y_train)    → entraîner")
print(".predict(X_test)          → prédire")
print(".feature_importances_     → quelles variables comptent le plus")
print("classification_report()   → précision, rappel, F1-score")
